# Science Hallucination Detector
vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference. As a demonstration of that, in this notebook, we show how vLLM-Hook enables *scientific-claim hallucination detector*.

**Paper**: [Detecting Hallucinations in Scientific Claims by Combining Prompting Strategies and Internal State Classification](https://aclanthology.org/2025.sdp-1.30/).<br />
**Authors**: Yupeng Cao, Chun-Nam Yu, K.P. Subbalakshmi.<br />
**"TL;DR"**: Each (claim, reference) pair is fed to an LLM with a few-shot prompt asking for a justification and classification (*entailment*, *contradiction*, or *unverifiable*). The model's hidden state at the last response token is then classified by a small logistic-regression model trained in advance.

### Installation

The cell below installs MIA. Run from a clone of the repository (this notebook sits in `notebooks/`), it installs the validated dependencies from `requirement.txt` and the checkout in editable mode; anywhere else it installs `vllm-hook` from PyPI.<br />
It is not necessary if the package is already installed in this environment.

In [ ]:
from pathlib import Path

# In a clone, this notebook is in <repo>/notebooks/ next to requirement.txt.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
REQ_FILE = REPO_ROOT / "requirement.txt"

if REQ_FILE.exists() and (REPO_ROOT / "pyproject.toml").exists():
    print("Repo checkout found at", REPO_ROOT)
    %pip install -r "{REQ_FILE}"
    %pip uninstall -y torchcodec
    %pip install --no-deps -e "{REPO_ROOT}"
else:
    print("No repo checkout found: installing vllm-hook from PyPI")
    %pip install vllm-hook

### Importing the MIA LLM
MIA provides `MiaLLM`, a wrapper around `vllm.LLM` that arms a capture or steering worker and runs an analyzer on what it captures.
We import it here together with `config_path`, which finds the example configs and steering vectors whether MIA came from a clone or from PyPI:

In [ ]:
from vllm import SamplingParams, TokensPrompt
from mia import MiaLLM, config_path

### Environment & multiprocessing setup

In [ ]:
import json
import os
import multiprocessing as mp
import torch
mp.set_start_method("spawn", force=True)
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

### Helpers: dataset fetcher and prompt builder
The SciHal-2025 train and test splits are committed to [InfintyLab/SciHal-Challenge](https://github.com/InfintyLab/SciHal-Challenge) on GitHub. We fetch each split once and cache it locally.

In [3]:
SciHal_url = (
    "https://raw.githubusercontent.com/InfintyLab/SciHal-Challenge/"
    "main/data/dataset/"
)

def load_scihal_split(cache_dir: str, filename: str) -> list:
    """Download SciHal-Challenge dataset if no cache in local."""
    local = os.path.join(cache_dir, filename)
    if not os.path.exists(local):
        from urllib.request import urlretrieve
        print(f"Downloading SciHal to {local}")
        urlretrieve(SciHal_url + filename, local)
    with open(local) as f:
        return json.load(f)

Following the original SciHal setup, we build a few-shot prefix from the train split with 2 examples per label.

In [4]:
def build_few_shot_middle(train_dataset: list, count_target: int = 2, total_target: int = 6) -> str:
    """Build few shot examples following https://github.com/InfintyLab/SciHal-Challenge/blob/97817c788bde330c61f8ed2fc750d83b19fa1590/llama3_finetunec3_just_task1.py#L141."""
    middle = ""
    count_dict = {"entailment": 0, "contradiction": 0, "unverifiable": 0}
    total = 0
    for x in train_dataset:
        label = x["label"]
        if count_dict.get(label, 0) >= count_target:
            continue
        my_input = "#Claim: " + x["claim"] + "\n #Reference: " + x["reference"]
        middle += (
            "Input: " + my_input + "\n\n"
            "Output:\n" + x["justification"] + "\n#Label: " + label + "\n\n"
        )
        count_dict[label] += 1
        total += 1
        if total == total_target:
            break
    return middle

Build SciHal prompt token IDs following the original authors' doubled-BOS practice: apply_chat_template returns a string already prefixed with <|begin_of_text|>, then tokenizer() with add_special_tokens=True prepends ANOTHER <|begin_of_text|>. The classifier was trained on hidden states extracted from this prefix.

In [5]:
PROMPT_TEMPLATE_PREFIX = (
    "\n"
    "You are a helpful assistant. Learn from the examples below and complete the task accordingly. \n"
    "\n"
    "### Task: Detect if the claims are well-supported by the references. Provide a justification and classify each example into three labels: entailment, contradiction, or unverifiable\n"
    "\n"
)

PROMPT_TEMPLATE_SUFFIX = (
    "\n"
    "\n"
    "### Now, apply the same pattern: \n"
    "\n"
    "Input: !INPUT!\n"
    "Output: \n"
)

def build_prompt_ids(tokenizer, few_shot_middle: str, claim: str, reference: str) -> list:
    my_input = "#Claim: " + claim + "\n #Reference: " + reference
    user_msg = few_shot_middle + PROMPT_TEMPLATE_SUFFIX.replace("!INPUT!", my_input)
    chat = [
        {"role": "system", "content": PROMPT_TEMPLATE_PREFIX},
        {"role": "user", "content": user_msg},
    ]
    message = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    return tokenizer(message, add_special_tokens=True).input_ids

### Initialize `MiaLLM`
Before we create the LLM instance, we need to specify the model and data type:

In [6]:
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
model = "meta-llama/Llama-3.1-8B-Instruct"

n_test = 9  # number of test items to classify

dtype_map = {
    'meta-llama/Llama-3.1-8B-Instruct': torch.float16,
}

We also need to provide a config file that specifies which decoder layer to capture and which classifier to use. For science hallucination detection, this config can be obtained from https://github.com/InfintyLab/SciHal-Challenge/blob/main/llama3_finetunec3_just_task1.py.

In [ ]:
from pathlib import Path

json_path = Path(config_path(f"hidden_states/{model.split('/')[-1]}.json"))

with open(json_path) as f:
    config = json.load(f)
# The classifier is trained separately (see examples/README.md); point MIA_SCIHAL_CLF at it,
# or set "scihal.clf_path" in the config.
clf_path = os.environ.get("MIA_SCIHAL_CLF", config["scihal"]["clf_path"])

Inside `capture_hs` and `science_hallucination` we define the desired behavior during model inference and after the model inference:
- `mia/workers/hs_capture_worker.py` defines that we need pre-final-norm residual at the requested token mode
- `mia/analyzers/science_hallucination_analyzer.py` runs the saved classifier on the model's final RMSNorm.

Now, we initialize the LLM:

In [ ]:
llm = MiaLLM(
    model=model,
    worker_name="capture_hs",
    analyzer_name="science_hallucination",
    config_file=str(json_path),
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    max_model_len=8192,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_prefix_caching=True,
    enable_hook=True,
)

### n_test Test cases
We use the first `n_test` items from `subtask1_test.json` and build the few-shot prefix from `subtask1_train_batch3.json`.

In [9]:
tokenizer = llm.tokenizer
train = load_scihal_split(cache_dir, "subtask1_train_batch3.json")
few_shot_middle = build_few_shot_middle(train)
test_cases = load_scihal_split(cache_dir, "subtask1_test.json")[:n_test]
prompt_ids_list = [
    build_prompt_ids(tokenizer, few_shot_middle, q["claim"], q["reference"]) for q in test_cases
]
print(f"prepared {len(prompt_ids_list)} prompts; example length = {len(prompt_ids_list[0])} tokens")

prepared 9 prompts; example length = 3398 tokens


We run a normal generation pass with hooks disabled to generate the response.

In [10]:
gen_outputs = llm.generate(
    [TokensPrompt(prompt_token_ids=ids) for ids in prompt_ids_list],
    SamplingParams(temperature=0.0, max_tokens=1024),
    use_hook=False,
)
response_token_ids = [list(gen_output.outputs[0].token_ids) for gen_output in gen_outputs]

Rendering prompts: 100%|████████████████████████████████████████████████| 9/9 [00:00<00:00, 87.60it/s]
Processed prompts: 100%|█| 9/9 [00:01<00:00,  6.56it/s, est. speed input: 23090.21 toks/s, output: 620


We then re-prefill with `prompt + response[:-2]` and capture the last-token hidden state following https://github.com/InfintyLab/SciHal-Challenge/blob/97817c788bde330c61f8ed2fc750d83b19fa1590/llama3_finetunec3_just_task1.py#L262.

In [11]:
capture_prompts = [
    TokensPrompt(prompt_token_ids=list(p) + list(r[:-2]))
    for p, r in zip(prompt_ids_list, response_token_ids)
]
output = llm.generate(capture_prompts, SamplingParams(temperature=0.0, max_tokens=1), save_to_disk=True)

Rendering prompts: 100%|██████████████████████████████████████████████| 9/9 [00:00<00:00, 3910.17it/s]
Processed prompts: 100%|█| 9/9 [00:00<00:00, 150.22it/s, est. speed input: 549754.40 toks/s, output: 1


Finally we classify with the science_hallucination analyzer.

In [12]:
LABEL_NAMES = ["entailment", "contradiction", "unverifiable"]
spec = {
    "label_names": LABEL_NAMES,
    "clf_path": clf_path,
    "model_id": model,
}
stats = llm.analyze(probes=getattr(output[0], "probes", None), analyzer_spec=spec)

labels = stats["prediction_labels"]
print("=" * 50)
for case, label in zip(test_cases, labels):
    print(f"classifier label: {label}")

/proj/dmfexp/irene/miniconda3/envs/vllm_hook_env/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Fetching 6 files: 100%|██████████████████████████████████████████████| 6/6 [00:00<00:00, 63072.24it/s]


classifier label: contradiction
classifier label: contradiction
classifier label: entailment
classifier label: contradiction
classifier label: unverifiable
classifier label: entailment
classifier label: entailment
classifier label: contradiction
classifier label: entailment


### (Optional) User can also try out rpc path which allows easier debugging

In [13]:
output = llm.generate(capture_prompts, SamplingParams(temperature=0.0, max_tokens=1), save_to_disk=False)
stats = llm.analyze(probes=getattr(output[0], "probes", None), analyzer_spec=spec)

labels = stats["prediction_labels"]
print("=" * 50)
for case, label in zip(test_cases, labels):
    print(f"classifier label: {label}")

Rendering prompts: 100%|██████████████████████████████████████████████| 9/9 [00:00<00:00, 4005.60it/s]
Processed prompts: 100%|█| 9/9 [00:00<00:00, 168.92it/s, est. speed input: 617678.58 toks/s, output: 1


classifier label: contradiction
classifier label: contradiction
classifier label: entailment
classifier label: contradiction
classifier label: unverifiable
classifier label: entailment
classifier label: entailment
classifier label: contradiction
classifier label: entailment
